# 01 Non-muscle model edits

Run this notebook first. It applies edits that do not modify muscle properties and writes a handoff model for downstream notebooks.

In [ ]:
import opensim as osim
import re

from rathindlimb.processing import update_model
from rathindlimb.project import project_paths

paths = project_paths()
model_dir = paths.model_dir
xml_dir = paths.xml_dir
pipeline_dir = paths.pipeline_dir
pipeline_dir.mkdir(parents=True, exist_ok=True)

source_model_file = model_dir / 'rat_hindlimb.osim'
non_muscle_out = pipeline_dir / 'rat_hindlimb_non_muscle.osim'

with open(source_model_file, 'r') as file:
    file_content = file.read()

terms = ['pelvis', 'femur', 'tibia', 'foot', 'sacroiliac', 'hip', 'knee', 'ankle']
for term in terms:
    file_content = re.sub(rf'{term}(?!_r)', f'{term}_r', file_content)

file_content = file_content.replace('.vtp', '.stl')

with open(non_muscle_out, 'w') as file:
    file.write(file_content)

model = osim.Model(str(non_muscle_out))

#TODO: Remove defaults block


[info] Updating Model file from 40000 to latest format...
[info] Loaded model RatHindlimbRight from file /home/hudson/RatHindlimb/models/osim/.pipeline/rat_hindlimb_non_muscle.osim
[warning] Couldn't find file 'ground.stl'.
[warning] Couldn't find file 'spine.stl'.
[warning] Couldn't find file 'pelvis_r.stl'.
[warning] Couldn't find file 'femur_r.stl'.
[warning] Couldn't find file 'tibia_r.stl'.
[warning] Couldn't find file 'foot_r.stl'.


In [6]:
coords_to_lock = ['ankle_r_add', 'ankle_r_int', 'sacroiliac_r_flx']
model.initSystem()
for coord_name in coords_to_lock:
    coord = model.updCoordinateSet().get(coord_name)
    coord.set_locked(True)

new_frame = [-0.00057, 0.0399598, 0.0038162]
rotation3 = [0.26179900000000000393] * 14
translation1 = [-0.0052385299999999999226, -0.0046464799999999997077, -0.0040425699999999996706, -0.0034725400000000000884, -0.0029796499999999999202, -0.0025907899999999999333, -0.0022994299999999998768, -0.0021100099999999998371, -0.001995639999999999914, -0.0019320699999999999246, -0.0018704800000000001026, -0.0017793500000000000722, -0.0016382700000000000474, -0.0014133999999999999862]
translation2 = [-0.034168400000000001548, -0.03425389999999999685, -0.034185599999999996546, -0.033972299999999996944, -0.033651100000000003232, -0.033278500000000002523, -0.032889799999999996816, -0.032550400000000000167, -0.032282499999999998697, -0.032116100000000001591, -0.032063200000000000034, -0.032110500000000000154, -0.032239700000000003077, -0.032414400000000002933]
translation3 = [0.0026030200000000001601, 0.0028034100000000001032, 0.0028536500000000001101, 0.002903750000000000081, 0.0028935800000000001971, 0.0027731800000000000894, 0.0027023500000000000645, 0.0026110500000000001763, 0.002469499999999999907, 0.0024171399999999999136, 0.0024041000000000001605, 0.0023809000000000000302, 0.002496410000000000122, 0.0026910400000000000119]

knee = osim.CustomJoint.safeDownCast(model.getJointSet().get('knee_r'))
tibia_offset = osim.PhysicalOffsetFrame.safeDownCast(knee.getChildFrame())
tibia_offset.set_translation(osim.Vec3(new_frame[0], new_frame[1], new_frame[2]))

spatial_transform = knee.get_SpatialTransform()
spatial_transform.get_rotation3().set_function(osim.Constant(rotation3[0]))

simm1 = osim.SimmSpline.safeDownCast(spatial_transform.get_translation1().get_function())
for i, value in enumerate(translation1):
    simm1.setY(i, value)

simm2 = osim.SimmSpline.safeDownCast(spatial_transform.get_translation2().get_function())
for i, value in enumerate(translation2):
    simm2.setY(i, value)

simm3 = osim.SimmSpline.safeDownCast(spatial_transform.get_translation3().get_function())
for i, value in enumerate(translation3):
    simm3.setY(i, value)

marker_set_path = xml_dir / 'rat_hindlimb_unilateral_markers.xml'
marker_set = osim.MarkerSet(str(marker_set_path))
model.getMarkerSet().clearAndDestroy()
model.updateMarkerSet(marker_set)

model = update_model(model, non_muscle_out)
print(f'Wrote non-muscle handoff model: {non_muscle_out}')

[info] Updated markers in model RatHindlimbRight
[info] Loaded model RatHindlimbRight from file /home/hudson/RatHindlimb/models/osim/.pipeline/rat_hindlimb_non_muscle.osim
Wrote non-muscle handoff model: /home/hudson/RatHindlimb/models/osim/.pipeline/rat_hindlimb_non_muscle.osim
[warning] Couldn't find file 'ground.stl'.
[warning] Couldn't find file 'spine.stl'.
[warning] Couldn't find file 'pelvis_r.stl'.
[warning] Couldn't find file 'femur_r.stl'.
[warning] Couldn't find file 'tibia_r.stl'.
[warning] Couldn't find file 'foot_r.stl'.
